## Guided BackPropagation: Practice

As we saw in the theory, Guided Backpropagation is not only a method for the interpretation task in its own right, but also a building block of other methods.

Guided Backpropagation is a modification of the ordinary Vanilla Gradients algorithm. The main difference is that it puts the emphasis only on those gradients that were greater than zero.  They are passed only through positive activations, which makes it possible to get sharper and more interpretable visualisations.

In this practice we will look at how to apply the Guided Backpropagation method to a deep neural network in order to visualise the influence of different input features on the output prediction.

**Goals of the practice:**

- Understand the main concepts of the Guided Backpropagation method
- Learn to apply this method in practice
- Implement the Guided_backprop class yourself for the ResNet and AlexNet models

<img src="https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/assets/tim-foster-w-X64-Gjbclg-unsplash.jpg" alt="tim-foster-w-X64-Gjbclg-unsplash" border="0">

**Quiz 1.**Let us start by recalling what we have already covered. Answer — what is a Hook in pyTorch?

**Your answer:**


*   A mechanism for regularising gradients
*   A mechanism that lets users embed their own functions into the working process of the model
* A way to speed up training
* A way to normalise the loss function

We will need hooks to implement Guided backpropagation. Let us create a template of the class that will carry it out.

```
class Guided_backprop():

    def __init__(self, model):
        self.model = model
        self.image_reconstruction = None # Here will be the resulting activation map
        self.activation_maps = []  # here we will record f1, f2, ...
        self.model.eval()
        self.register_hooks()

    def register_hooks(self):
        def first_layer_hook_fn(module, grad_in, grad_out): # here will be the function for intercepting the first layer
            pass

        def forward_hook_fn(module, input, output): # here will be the function for the forward hook
            pass

        def backward_hook_fn(module, grad_in, grad_out): # here will be the function for the backward hook
            pass


    def visualize(self, input_image, target_class):

        model_output = self.model(input_image)
        pass
```

Let us load the image we are going to work with. To make it more interesting, we take an example that contains objects of two classes at once.

In [ ]:
# importing the necessary libraries
import torch
import requests
import numpy as np
from io import BytesIO
from torch import nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
# Loading the image
url = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/cat_and_dog.jpg'

image_bytes = requests.get(url).content
image = Image.open(BytesIO(image_bytes)) # Again we take one particular example x_0

# We transform the image to feed it to the trained network
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # Your code here
])


tensor = transform(image).unsqueeze(0).requires_grad_()

**Quiz 2** Finish the `transform` function. Add normalisation (`transforms.Normalize`) with the classic mean and standard deviation for ImageNet. As your answer, give the value at the coordinates `tensor[0, 1, 1, 1]`  

In [ ]:
# Your code here

Now let us load the model. We will work with resnet50.

In [ ]:
model = models.resnet50(pretrained=True)
model.eval();

**Quiz 3** Apply the model to the image. Which class did it predict? As your answer, give the number of the class.

In [ ]:
# Your code here

- Let us start filling in the class that implements Guided backpropagation. We begin with `forward_hook_fn(module, input, output)`. We need this hook to store the maps *obtained at particular layers* during the forward pass.

**Quiz 4** Following this logic, choose the correct implementation

```
`forward_hook_fn` in the trainer and write it into the class template.

 def forward_hook_fn(module, input, output):
            # Choose the function in the trainer
```

- Now let us think about the `backward hook`.

**Quiz 5** Looking at the implemented code, arrange the correct order of its steps on the platform.


```
def backward_hook_fn(module, grad_in, grad_out):
            grad = self.activation_maps.pop()
            grad[grad > 0] = 1
            
            positive_grad_out = torch.clamp(grad_out[0], min=0.0)
            new_grad_in = positive_grad_out * grad

            return (new_grad_in,)
```

- We will also need to save the feature map of the image ($f^0$). Usually the input image is considered, which can be defined as the input at the forward pass of the first layer of the model, that is:


```
def first_layer_hook_fn(module, grad_in, grad_out):
            self.image_reconstruction = grad_in[0]
```

**Excellent! Now let us implement all the transformations that our Guided Backpropagation has to perform.**

First of all, we need to extract all the components of the model.

`modules = list(self.model.named_modules())`

Next, we will apply the forward and the backward hook from the end to the beginning. For all transformations of the image except the first one, we will register the forward and backward hooks only after *certain* layers. Do you remember after which ones (the answer is in the theory of the lesson)?

We will take the hooks off the first layer only after passing through the preceding layers.

**Quiz 6**
Checking that a layer matches a certain type can be done with the `isinstance(object, type)` function. What should be in place of the type in our case?

Choose the answer in the trainer and complete the code.

```
for name, module in modules:
  if isinstance(module, # Your answer here):
    module.register_forward_hook(forward_hook_fn)
    module.register_backward_hook(backward_hook_fn)
```



        ##RESNET

        if 'resnet' in identify_model(self.model).lower():
          first_layer = modules[1][1]
          first_layer.register_backward_hook(first_layer_hook_fn)

        ###ALEXNET

        if 'alexnet' in identify_model(self.model).lower():
          first_layer = modules[1][1][0]
          first_layer.register_backward_hook(first_layer_hook_fn)

**At this step the main part of Guided Backpropagation is implemented. All that is left is to add the ability to visualise the prediction, and we are done!**

We will describe the visualisation with the following logic:

```

 def visualize(self, input_image, target_class):


        model_output = self.model(input_image) # we predict the class
        self.model.zero_grad() # we zero out the gradients
        pred_class = model_output.argmax().item() # we extract the model's prediction

        # we prepare a blank of 0s and 1s in order to do the backward pass over the parameters of the image we are interested in

        grad_target_map = torch.zeros(model_output.shape,
                                      dtype=torch.float)
        if target_class is not None:
            grad_target_map[0][target_class] = 1
        else:
            grad_target_map[0][pred_class] = 1

        model_output.backward(grad_target_map)

        result = self.image_reconstruction.data[0].permute(1,2,0) # we prepare the result for visualisation
        return result.numpy()
```

1. First of all, let us add the ability to build Guided backpropagation from any class we are interested in. For that the function will take a `target class` argument. By default we will build Guided backpropagation for the predicted class.
2. Secondly, all the results will be returned as a numpy array.

**Now let us put it all together and finish the class.**

In [ ]:
import torch
from torch import nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt



def identify_model(model):
    return model.__class__.__name__


class Guided_backprop():
    def __init__(self, model):
        self.model = model
        self.image_reconstruction = None # Here will be the resulting activation map
        self.activation_maps = []  # here we will record f1, f2, ...
        self.model.eval()
        self.register_hooks()

    def register_hooks(self):
        def first_layer_hook_fn(module, grad_in, grad_out):
            self.image_reconstruction = grad_in[0]

        def forward_hook_fn(module, input, output):
            self.activation_maps.append(output)

        def backward_hook_fn(module, grad_in, grad_out):
            grad = self.activation_maps.pop() # we take the last map in the list (f_l, f_l-1, f_l-2...)

            # the logical function at the forward pass, after ReLU
            # if the output value was not equal to 0, we make it one
            # and zero otherwise if the output value is positive, we set the value to 1,
            # and if the output value is negative, we set it to 0.
            grad[grad > 0] = 1

            #in grad_out[0] we will record the gradients for every feature map
            # only if the gradients were greater than zero
            positive_grad_out = torch.clamp(grad_out[0], min=0.0)

            #A logical AND over the results (equivalent to multiplication)
            new_grad_in = positive_grad_out * grad

            return (new_grad_in,)


        # AlexNet model
        modules = list(self.model.named_modules())

        # we move through the modules, extracting the maps at the forward and the backward pass
        # for ReLU
        for name, module in modules:
            if isinstance(module, nn.ReLU):
                module.register_forward_hook(forward_hook_fn)
                module.register_backward_hook(backward_hook_fn)

        ##RESNET

        if 'resnet' in identify_model(self.model).lower():
          first_layer = modules[1][1]
          first_layer.register_backward_hook(first_layer_hook_fn)

        ###ALEXNET

        if 'alexnet' in identify_model(self.model).lower():
          first_layer = modules[1][1][0]
          first_layer.register_backward_hook(first_layer_hook_fn)


    def visualize(self, input_image, target_class):


        model_output = self.model(input_image) # we predict the class
        self.model.zero_grad() # we zero out the gradients
        pred_class = model_output.argmax().item() # we extract the class label

        # we prepare a blank of 0s and 1s in order to do the backward pass over the parameters of the image we are interested in
        grad_target_map = torch.zeros(model_output.shape,
                                      dtype=torch.float)
        if target_class is not None:
            grad_target_map[0][target_class] = 1
        else:
            grad_target_map[0][pred_class] = 1

        model_output.backward(grad_target_map)

        result = self.image_reconstruction.data[0].permute(1,2,0) # we prepare the result for visualisation
        return result.numpy()

def normalize(image):
    "A function to improve the readability of the resulting map"
    norm = (image - image.mean())/image.std()
    norm = norm * 0.1
    norm = norm + 0.5
    norm = norm.clip(0, 1)
    return norm

In [ ]:
guided_bp = Guided_backprop(model)
result = guided_bp.visualize(tensor, None)

result = normalize(result)
plt.imshow(result)
plt.show()

Build the map for the class tiger cat (282). Has it changed?

In [ ]:
guided_bp = Guided_backprop(model)
result1 = guided_bp.visualize(tensor, 282)

result1 = normalize(result1)  # карта строится для класса 282 — её и нормализуем
plt.imshow(result1)
plt.show()

Build Guided backprop for any random class. Does the result change much?

In [ ]:
guided_bp = Guided_backprop(model)
result = guided_bp.visualize(tensor, 100)

result = normalize(result)
plt.imshow(result)
plt.show()